# P4 — DistilBERT Fine-tuning on PubMed RCT

Fine-tunes `distilbert-base-uncased` on PubMed 200k RCT sentence classification.

**Labels:** BACKGROUND | OBJECTIVE | METHODS | RESULTS | CONCLUSIONS  
**Runtime:** Kaggle T4 GPU (~45 min, 3 epochs)  
**Expected accuracy:** ≥ 85%

## Kaggle Secrets required (Settings → Add secret)
| Secret | Value |
|--------|-------|
| `RGW_ENDPOINT` | `http://100.82.75.34` (Tailscale IP of quick-thrush) |
| `RGW_ACCESS_KEY` | from `pass homelab/rgw/access-key` on your laptop |
| `RGW_SECRET_KEY` | from `pass homelab/rgw/secret-key` on your laptop |

In [ ]:
!pip install -q 'transformers>=4.41.0' 'datasets' 'evaluate' 'accelerate'
# boto3 is pre-installed on Kaggle with compatible botocore — do not reinstall

In [ ]:
import json, os, time
from pathlib import Path
import evaluate
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainingArguments,
)
print(f"PyTorch: {torch.__version__}, GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {torch.cuda.get_device_name(0)}")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 128
BATCH_SIZE = 64
EPOCHS     = 3
LR         = 2e-5
OUTPUT_DIR = "/kaggle/working/pubmed_rct_model"

# armanc/pubmed-rct20k uses lowercase string labels
LABEL2ID = {"background": 0, "objective": 1, "methods": 2, "results": 3, "conclusions": 4}
ID2LABEL = {v: k.upper() for k, v in LABEL2ID.items()}

In [ ]:
print("Loading dataset...")
# armanc/pubmed-rct20k: 20k abstracts (~177k sentences), same task as 200k RCT
# Chosen over pietrolesci/pubmed-200k-rct (2.27M sentences) because one epoch
# on the full dataset takes ~6 hours on a T4 — well beyond Kaggle's session limit.
dataset = load_dataset("armanc/pubmed-rct20k")
print(dataset)
print("Sample:", dataset["train"][0])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)
    out["labels"] = [LABEL2ID[l.lower()] for l in batch["label"]]
    return out

tokenized = dataset.map(
    tokenize, batched=True,
    remove_columns=["text", "abstract_id", "sentence_id", "label"]
)
tokenized.set_format("torch")
print("Tokenisation done.")
print("Train size:", len(tokenized["train"]), "| Val size:", len(tokenized["validation"]))

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=5, id2label=ID2LABEL, label2id=LABEL2ID,
)
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc    = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1     = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    f1_per = f1_metric.compute(predictions=preds, references=labels, average=None)["f1"]
    result = {"accuracy": acc, "f1_macro": f1}
    for i, label in ID2LABEL.items():
        result[f"f1_{label.lower()}"] = float(f1_per[i])
    return result

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available(),
    logging_steps=500,
    report_to="none",
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,        # renamed from 'tokenizer' in transformers>=4.47
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

t0 = time.time()
trainer.train()
print(f"Training complete in {(time.time()-t0)/60:.1f} min")

In [ ]:
results = trainer.evaluate(tokenized["test"])
print(json.dumps({k: round(v, 4) for k, v in results.items() if "runtime" not in k}, indent=2))
assert results["eval_accuracy"] >= 0.85, f"Accuracy {results['eval_accuracy']:.3f} < 0.85"

In [ ]:
best = Path(OUTPUT_DIR) / "best_model"
trainer.save_model(str(best))
tokenizer.save_pretrained(str(best))
(best / "metrics.json").write_text(json.dumps({
    "model": MODEL_NAME, "dataset": "pubmed_rct_200k", "epochs": EPOCHS,
    "accuracy": results["eval_accuracy"], "f1_macro": results["eval_f1_macro"],
    **{k.replace("eval_", ""): v for k, v in results.items() if "f1_" in k and "macro" not in k},
}, indent=2))
print("Saved:", best)
!ls -lh {best}

In [ ]:
import boto3
from botocore.client import Config
from kaggle_secrets import UserSecretsClient

sec = UserSecretsClient()
s3 = boto3.client(
    "s3",
    endpoint_url=sec.get_secret("RGW_ENDPOINT"),
    aws_access_key_id=sec.get_secret("RGW_ACCESS_KEY"),
    aws_secret_access_key=sec.get_secret("RGW_SECRET_KEY"),
    config=Config(signature_version="s3v4"),
)

BUCKET, PREFIX = "nlp-models", "pubmed-rct/v1"
existing = {b["Name"] for b in s3.list_buckets().get("Buckets", [])}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)

for path in best.rglob("*"):
    if path.is_file():
        key = f"{PREFIX}/{path.relative_to(best)}"
        print(f"  {path.name} → s3://{BUCKET}/{key}")
        s3.upload_file(str(path), BUCKET, key)

print("\nDone. Files in bucket:")
for obj in s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX).get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']/1e6:.1f} MB)")